# JONICA26 — Entrenar con el muñeco y convertir a `.hef`

Todo el camino en un solo notebook, de arriba a abajo:

```
Roboflow (pareysemaforo + muñeco)  ->  dataset unido, 3 clases
   -> entrenar YOLOv8n (GPU)  ->  .pt  ->  .onnx  ->  .hef (Hailo-8)  ->  verificar
```

Lo que nos costó la primera vez ya está resuelto adentro: versiones del DFC y
del Model Zoo, el parche de torch, el `ir_version`, la variable `USER`, la
salida NMS transpuesta. **No hace falta tocar nada salvo la celda de ajustes.**

### Antes de empezar (una sola vez)

1. **Entorno con GPU:** *Entorno de ejecución → Cambiar tipo de entorno →
   T4 GPU*. Entrenar en CPU tarda horas. La compilación del `.hef` usa CPU,
   pero corre igual en este entorno.
2. **El wheel del DFC en Google Drive**, en la carpeta `jonica26/`
   (`Mi unidad/jonica26/hailo_dataflow_compiler-3.33.x-....whl`). Es el
   mismo que usamos para el modelo actual. Si no está, el paso 5 lo pide a mano.
3. **(Opcional) La API key de Roboflow como secreto de Colab:** ícono de la
   llave 🔑 a la izquierda → `ROBOFLOW_API_KEY`. Si no, la pide al bajar.

### El dataset del muñeco en Roboflow

- Proyecto de **Object Detection**. Puede ser el mismo `pareysemaforo` o uno
  aparte (`muneco`): el notebook junta los que le digas.
- La clase se llama **`muneco`**. Si alguien escribe `Muñeco` o `MUNECO` el
  notebook lo reconoce igual, pero cualquier otro nombre lo frena con un aviso.
- Si en una foto aparece un PARE o un semáforo, **etiquetarlo también** con
  el mismo nombre (`PARE`, `semaforoverde`).
- Al generar la versión: *Preprocessing* → Auto-Orient y Resize 640×640
  (Stretch). **Sin aumentos**: los hace el entrenamiento, incluido dar vuelta
  la imagen (la cámara del robot va montada al revés y el modelo la ve así).

### Si Colab se desconecta

El entrenamiento se guarda en Drive en cada época. Volver a correr el paso 0,
reiniciar si lo pide el paso 1, correr el paso 2 (dataset) y el 3 retoma donde
quedó. Si el `.pt` ya está en Drive, el paso 3 no reentrena.

## Paso 0 — Ajustes

**Lo único que se edita.** Se guarda en un archivo para que sobreviva a los
reinicios del entorno: cada celda lo vuelve a leer.

In [ ]:
%%writefile /content/ajustes.py
import os

# Datasets de Roboflow que se juntan: (proyecto, version). La version es el
# numero que aparece en Roboflow al generarla. El workspace es el de la API
# key; si un proyecto esta en otro, escribir "workspace/proyecto".
ROBOFLOW = [
    ("pareysemaforo", 3),
    ("muneco", 1),
]

# Orden FIJO de las clases del modelo nuevo. muneco va al final para que PARE y
# semaforoverde conserven su numero. Este orden termina en labels.txt y en
# ETIQUETAS de source/config.py.
CLASES = ["PARE", "semaforoverde", "muneco"]

NOMBRE = "yolov8n_pare_semaforo_muneco"
EPOCAS = 100
REENTRENAR = False  # True: entrena de cero aunque ya haya un .pt en Drive

DRIVE = "/content/drive/MyDrive/jonica26"

PY = "/content/hailo_env/bin/python"
PIP = "/content/hailo_env/bin/pip"
HAILOMZ = "/content/hailo_env/bin/hailomz"
DATOS = "/content/datos"
PT = f"/content/{NOMBRE}.pt"
ONNX = f"/content/{NOMBRE}.onnx"
HEF = f"/content/{NOMBRE}.hef"

# Colab corre como root sin USER, y el compilador de Hailo la lee.
os.environ["USER"] = "root"
# torch >= 2.6 carga los .pt en modo restringido y Ultralytics 8.2 no lo sabe.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"


def montar_drive():
    if not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    os.makedirs(DRIVE, exist_ok=True)

In [ ]:
exec(open("/content/ajustes.py").read())
import glob
import platform
import subprocess

assert platform.machine() == "x86_64", "El compilador de Hailo necesita x86_64"
flags = subprocess.run(["grep", "-m1", "flags", "/proc/cpuinfo"],
                       capture_output=True, text=True).stdout
assert "avx" in flags, "Esta CPU no tiene AVX: el compilador de Hailo no corre"

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NINGUNA")
if not gpu:
    print("  AVISO: sin GPU entrenar tarda horas.")
    print("  Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU")

ram_gb = int(open("/proc/meminfo").readline().split()[1]) / 1024 / 1024
print(f"RAM: {ram_gb:.1f} GB")

montar_drive()
wheels = sorted(glob.glob(DRIVE + "/hailo_dataflow_compiler-3.33*.whl"))
if wheels:
    print("Wheel del DFC:", wheels[-1])
else:
    print(f"AVISO: no hay wheel del DFC 3.33 en {DRIVE}. El paso 5 lo va a pedir a mano.")

## Paso 1 — Instalar

Después de esta celda **hay que reiniciar** (*Entorno de ejecución →
Reiniciar sesión*): la instalación cambia numpy con numpy ya cargado, y si no
se reinicia el error aparece mucho más adelante y no se entiende. Reiniciar no
borra nada de lo instalado ni lo de Drive.

In [ ]:
# onnxscript lo usa torch al exportar aunque Ultralytics no lo declare.
!pip install -q ultralytics==8.2.103 roboflow onnx onnxruntime onnxslim onnxscript

print("\n" + "=" * 60)
print("REINICIA EL ENTORNO:  Entorno de ejecucion -> Reiniciar sesion")
print("Despues segui desde el paso 2 (no hace falta repetir 0 ni 1).")
print("=" * 60)

## Paso 2 — Bajar y juntar los datasets

Baja cada versión de Roboflow y las une en `/content/datos` con el orden de
clases de `CLASES`. Roboflow ordena las clases alfabéticamente y cada proyecto
puede tener otro orden: acá se renumeran por nombre, así nunca se mezclan.

Las etiquetas hechas con Smart Polygon se pasan a cajas.

In [ ]:
exec(open("/content/ajustes.py").read())
import getpass
import shutil
import unicodedata
from pathlib import Path

import yaml
from roboflow import Roboflow

try:
    from google.colab import userdata
    clave = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    clave = None
rf = Roboflow(api_key=clave or getpass.getpass("API key de Roboflow: "))


def normalizar(nombre):
    """'Muñeco', 'MUNECO' y 'muneco' son la misma clase."""
    sin_tildes = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    return "".join(c for c in sin_tildes.lower() if c.isalnum())


INDICE = {normalizar(c): i for i, c in enumerate(CLASES)}
SPLITS = ("train", "valid", "test")
EXTENSIONES = (".jpg", ".jpeg", ".png")

destino = Path(DATOS)
if destino.exists():
    shutil.rmtree(destino)
for s in SPLITS:
    (destino / s / "images").mkdir(parents=True)
    (destino / s / "labels").mkdir(parents=True)

imagenes = {s: 0 for s in SPLITS}
instancias = {s: [0] * len(CLASES) for s in SPLITS}

for ref, version in ROBOFLOW:
    workspace, _, proyecto = ref.rpartition("/")
    ws = rf.workspace(workspace) if workspace else rf.workspace()
    bajado = Path(ws.project(proyecto).version(version).download(
        "yolov8", location=f"/content/roboflow/{proyecto}-{version}", overwrite=True).location)

    nombres = yaml.safe_load((bajado / "data.yaml").read_text())["names"]
    if isinstance(nombres, dict):
        nombres = [nombres[k] for k in sorted(nombres)]
    remapeo = {i: INDICE.get(normalizar(n)) for i, n in enumerate(nombres)}
    print(f"{proyecto} v{version}: clases {nombres}")

    for s in SPLITS:
        carpeta = bajado / s / "images"
        if not carpeta.exists():
            continue
        for img in sorted(carpeta.iterdir()):
            if img.suffix.lower() not in EXTENSIONES:
                continue
            etiqueta = bajado / s / "labels" / (img.stem + ".txt")
            lineas = []
            for linea in (etiqueta.read_text().splitlines() if etiqueta.exists() else []):
                partes = linea.split()
                if not partes:
                    continue
                nueva = remapeo.get(int(partes[0]))
                if nueva is None:
                    raise ValueError(
                        f"La clase '{nombres[int(partes[0])]}' de {proyecto} no esta en "
                        f"CLASES {CLASES}. Renombrarla en Roboflow o agregarla en el paso 0.")
                valores = [float(v) for v in partes[1:]]
                if len(valores) > 4:  # poligono -> caja que lo contiene
                    xs, ys = valores[0::2], valores[1::2]
                    valores = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2,
                               max(xs) - min(xs), max(ys) - min(ys)]
                lineas.append(f"{nueva} " + " ".join(f"{v:.6f}" for v in valores))
                instancias[s][nueva] += 1
            # El prefijo evita que dos proyectos con el mismo nombre de foto se pisen.
            nombre = f"{proyecto}{version}_{img.stem}"
            shutil.copy(img, destino / s / "images" / (nombre + img.suffix))
            (destino / s / "labels" / (nombre + ".txt")).write_text(
                "".join(l + "\n" for l in lineas))
            imagenes[s] += 1

(destino / "data.yaml").write_text(yaml.safe_dump({
    "path": str(destino),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": dict(enumerate(CLASES)),
}, sort_keys=False))

print(f"\n{'':8}{'imagenes':>9}" + "".join(f"{c:>15}" for c in CLASES))
for s in SPLITS:
    print(f"{s:8}{imagenes[s]:>9}" + "".join(f"{n:>15}" for n in instancias[s]))

for i, c in enumerate(CLASES):
    if instancias["train"][i] == 0:
        print(f"\nERROR: no hay ninguna etiqueta de '{c}' en train. Revisar Roboflow.")
    elif instancias["valid"][i] == 0:
        print(f"\nAVISO: no hay '{c}' en valid: no se va a poder medir esa clase.")

## Paso 3 — Entrenar

Arranca de `yolov8n.pt` (preentrenado en COCO), como el modelo del año
pasado. Con una T4 tarda más o menos **30 a 60 minutos** según cuántas fotos
haya. Se guarda en Drive (`jonica26/entrenamientos/`) en cada época.

`flipud=0.5`: la mitad de las veces da vuelta la imagen de arriba a abajo, y
junto con el espejo izquierda-derecha que ya viene activado, una de cada
cuatro queda girada 180°. Hace falta porque la cámara del robot va montada al
revés y el modelo recibe la imagen así: sólo el video se endereza.

In [ ]:
exec(open("/content/ajustes.py").read())
import shutil
from pathlib import Path

from ultralytics import YOLO, settings

# Colab trae wandb instalado y Ultralytics lo activa solo; wandb rechaza que el
# proyecto sea una ruta y corta el entrenamiento antes de empezar. Vaciar sus
# callbacks cubre el caso en que un intento anterior ya lo cargo en memoria.
settings.update({"wandb": False})
import ultralytics.utils.callbacks.wb as _wb
_wb.callbacks.clear()

montar_drive()
corridas = Path(DRIVE) / "entrenamientos"
corrida = corridas / NOMBRE
en_drive = Path(DRIVE) / f"{NOMBRE}.pt"
ultimo = corrida / "weights" / "last.pt"

if en_drive.exists() and not REENTRENAR:
    print(f"Ya hay un modelo entrenado en Drive: {en_drive}")
    print("No se reentrena. Para entrenar de nuevo: REENTRENAR = True en el paso 0.")
else:
    if ultimo.exists() and not REENTRENAR:
        print("Hay un entrenamiento cortado a la mitad: se retoma desde", ultimo)
        try:
            YOLO(str(ultimo)).train(resume=True)
        except AssertionError as e:  # ya habia terminado todas las epocas
            print(e)
    else:
        if corrida.exists():
            shutil.rmtree(corrida)
        YOLO("yolov8n.pt").train(
            data=f"{DATOS}/data.yaml",
            epochs=EPOCAS,
            imgsz=640,
            batch=16,
            patience=25,  # corta solo si deja de mejorar
            flipud=0.5,
            project=str(corridas),
            name=NOMBRE,
            exist_ok=True,
        )
    shutil.copy(corrida / "weights" / "best.pt", en_drive)
    print("\nModelo guardado en Drive:", en_drive)

shutil.copy(en_drive, PT)
print("Listo:", PT)

## Paso 4 — ¿Aprendió?

Métricas sobre `valid` (fotos que no vio al entrenar) y algunas detecciones
para mirar con los ojos, derechas y dadas vuelta.

Como referencia: **recall del muñeco ≥ 85 %** está bien para la pista. Si da
mucho menos, conviene sumar fotos del caso que falla (lejos, con poca luz...)
antes de gastar media hora convirtiendo.

In [ ]:
exec(open("/content/ajustes.py").read())
import random
from pathlib import Path

from IPython.display import display
from PIL import Image
from ultralytics import YOLO

modelo = YOLO(PT)
assert list(modelo.names.values()) == CLASES, f"Clases del modelo: {modelo.names}"

m = modelo.val(data=f"{DATOS}/data.yaml", split="val", verbose=False)
print(f"\n{'clase':15}{'precision':>10}{'recall':>8}{'mAP50':>8}")
for k, ci in enumerate(m.box.ap_class_index):
    print(f"{CLASES[ci]:15}{m.box.p[k]:>10.1%}{m.box.r[k]:>8.1%}{m.box.ap50[k]:>8.1%}")

# Fotos de valid con muneco: izquierda derecha, derecha girada 180 grados.
indice_muneco = CLASES.index("muneco")
valid = Path(DATOS) / "valid"
con_muneco = [
    img for img in sorted((valid / "images").iterdir())
    if any(l.split()[0] == str(indice_muneco)
           for l in (valid / "labels" / (img.stem + ".txt")).read_text().splitlines())
]
random.seed(0)
for ruta in random.sample(con_muneco, min(4, len(con_muneco))):
    original = Image.open(ruta).convert("RGB")
    lado = []
    for img in (original, original.rotate(180)):
        r = modelo.predict(img, imgsz=640, conf=0.25, verbose=False)[0]
        lado.append(Image.fromarray(r.plot()[..., ::-1]).resize((400, 400)))
    par = Image.new("RGB", (800, 400))
    par.paste(lado[0], (0, 0))
    par.paste(lado[1], (400, 0))
    display(par)
if not con_muneco:
    print("No hay fotos con muneco en valid para mostrar.")

## Paso 5 — Instalar el compilador de Hailo

Versiones fijas, no tocar: **DFC 3.33** + **Model Zoo v2.18**, que es lo que
corresponde a HailoRT 4.23 de la Raspberry. Un `.hef` de otra versión compila
bien y después no carga. Tarda unos 5–10 minutos.

In [ ]:
exec(open("/content/ajustes.py").read())
import glob
import pathlib

montar_drive()
wheels = sorted(glob.glob(DRIVE + "/hailo_dataflow_compiler-3.33*.whl"))
if wheels:
    WHEEL = wheels[-1]
else:
    from google.colab import files
    print("Elegi el hailo_dataflow_compiler-3.33.x-...whl")
    WHEEL = "/content/" + next(iter(files.upload()))
print("Wheel:", WHEEL)

# El DFC pide Python 3.8-3.10: va en un venv aparte con 3.10.
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3.10 python3.10-venv python3.10-dev build-essential > /dev/null
!python3.10 -m venv /content/hailo_env
!{PIP} install -q --upgrade pip setuptools wheel
!{PIP} install -q "{WHEEL}"

# Model Zoo v2.18: la rama principal es la 5.x, que ya no soporta hailo8.
# Desde la URL git falla al construir: se clona y se instala sin aislamiento.
!rm -rf /content/model_zoo
!git clone -q --depth 1 --branch v2.18 https://github.com/hailo-ai/hailo_model_zoo.git /content/model_zoo
!{PIP} install -q --no-build-isolation -e /content/model_zoo

# El Model Zoo importa al arrancar un backend de torch que espera una clase que
# el DFC 3.33 no tiene. La calibracion no lo usa.
torch_infer = pathlib.Path("/content/model_zoo/hailo_model_zoo/core/infer/torch_infer.py")
if torch_infer.exists():
    torch_infer.rename(torch_infer.with_suffix(".py.desactivado"))
!find /content/model_zoo -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null

!{PY} -c "from hailo_sdk_client import ClientRunner; print('DFC OK')"
!{HAILOMZ} compile --help 2>&1 | grep -q "hailo8" && echo "Model Zoo OK: soporta hailo8" || echo "ERROR: este Model Zoo no soporta hailo8"

## Paso 6 — Exportar a ONNX

Opset 11 y **sin NMS**: el NMS se compila dentro del `.hef` para que corra en
el chip.

In [ ]:
exec(open("/content/ajustes.py").read())
import onnx
import torch
from ultralytics import YOLO

# Torch nuevo exporta con "dynamo", que ignora opset=11 y genera 18, que el
# parser de Hailo no acepta. Se fuerza el exportador clasico.
_export = torch.onnx.export


def _export_clasico(*args, **kwargs):
    kwargs["dynamo"] = False
    return _export(*args, **kwargs)


torch.onnx.export = _export_clasico
try:
    modelo = YOLO(PT)
    assert list(modelo.names.values()) == CLASES, f"Clases del modelo: {modelo.names}"
    salida = modelo.export(format="onnx", opset=11, imgsz=640, simplify=True)
finally:
    torch.onnx.export = _export

m = onnx.load(salida)
opset = [o.version for o in m.opset_import if o.domain in ("", "ai.onnx")][0]
assert opset <= 17, f"Opset {opset}: Hailo acepta 8 y 11-17"
# El DFC valida hasta ir_version 10; el grafo no cambia.
if m.ir_version > 10:
    m.ir_version = 10
onnx.save(m, ONNX)
print(f"{ONNX}  (opset {opset}, ir_version {m.ir_version})")

## Paso 7 — Imágenes de calibración

Para cuantizar a INT8 el compilador mide los rangos de cada capa con imágenes
reales. Salen de `train` y `test`; **`valid` queda reservado** para el paso 9,
así la verificación no da un número optimista.

In [ ]:
exec(open("/content/ajustes.py").read())
import random
import shutil
from pathlib import Path

from PIL import Image

fuente = [p for s in ("train", "test") for p in sorted((Path(DATOS) / s / "images").iterdir())]
random.seed(42)
random.shuffle(fuente)

calib = Path("/content/calib")
if calib.exists():
    shutil.rmtree(calib)
calib.mkdir()
for i, ruta in enumerate(fuente[:1024]):
    Image.open(ruta).convert("RGB").resize((640, 640)).save(calib / f"calib_{i:05d}.jpg")
print("Imagenes de calibracion:", len(list(calib.iterdir())))

## Paso 8 — Compilar el `.hef`

**`hailo8`** (26 TOPS), no `hailo8l`: no son intercambiables. Tarda entre
**15 y 40 minutos**.

In [ ]:
exec(open("/content/ajustes.py").read())
import shutil
from pathlib import Path

# Restos de corridas anteriores: aca y en el paso 9 se toma el mas nuevo.
for viejo in list(Path("/content").glob("*.har")) + list(Path("/content").glob("*.hef")):
    viejo.unlink()

!cd /content && {HAILOMZ} compile yolov8n --ckpt {ONNX} --hw-arch hailo8 --calib-path /content/calib --classes {len(CLASES)}

hefs = sorted(Path("/content").glob("*.hef"), key=lambda p: p.stat().st_mtime)
assert hefs, "No se genero el .hef: mirar el error de arriba"
shutil.copy(hefs[-1], HEF)
montar_drive()
shutil.copy(HEF, DRIVE)
print(f"\n{HEF}  ({Path(HEF).stat().st_size / 1e6:.1f} MB), copia en Drive")

## Paso 9 — Verificar la cuantización ⚠️ no saltear

La cuantización a INT8 puede **compilar bien y detectar peor**, sin ningún
error. Se compara lo que detecta el `.pt` contra el modelo cuantizado (en el
emulador del compilador) sobre fotos de `valid`.

Recall ≥ 90 % → OK. Entre 75 y 90 % → probar con cuidado. Menos → algo salió mal.

In [ ]:
exec(open("/content/ajustes.py").read())
import json
import random
import shutil
from pathlib import Path

from PIL import Image
from ultralytics import YOLO

modelo = YOLO(PT)
prueba = sorted((Path(DATOS) / "valid" / "images").iterdir())
random.seed(1)
prueba = random.sample(prueba, min(100, len(prueba)))

muestras = Path("/content/muestras")
if muestras.exists():
    shutil.rmtree(muestras)
muestras.mkdir()

referencia = {}
for ruta in prueba:
    nombre = ruta.stem + ".jpg"
    img = Image.open(ruta).convert("RGB").resize((640, 640))
    img.save(muestras / nombre)
    r = modelo.predict(img, imgsz=640, conf=0.25, verbose=False)[0]
    referencia[nombre] = [
        {"nombre": CLASES[int(c.cls[0])], "confianza": float(c.conf[0]), "bbox": c.xyxyn[0].tolist()}
        for c in r.boxes
    ]
Path("/content/referencia_pt.json").write_text(json.dumps(referencia, indent=2))
print("Detecciones del .pt:", sum(len(v) for v in referencia.values()))

Path("/content/emular.py").write_text("""
import json, sys
import numpy as np
from pathlib import Path
from PIL import Image
from hailo_sdk_client import ClientRunner, InferenceContext

har = sorted(Path("/content").glob("*.har"), key=lambda p: p.stat().st_mtime)
if not har:
    sys.exit("No hay .har: correr el paso 8")
runner = ClientRunner(har=str(har[-1]))
nombres = sorted(p.name for p in Path("/content/muestras").glob("*.jpg"))
lote = np.stack([np.asarray(Image.open("/content/muestras/" + n), dtype=np.uint8) for n in nombres])
with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
    salida = runner.infer(ctx, lote)
np.save("/content/salida_cuantizada.npy", np.array(salida, dtype=object), allow_pickle=True)
json.dump(nombres, open("/content/muestras.json", "w"))
print("Emulacion OK:", len(nombres), "imagenes")
""")
!cd /content && {PY} /content/emular.py

In [ ]:
exec(open("/content/ajustes.py").read())
import json
from collections import Counter

import numpy as np

CONF, IOU_MIN = 0.25, 0.5
referencia = json.load(open("/content/referencia_pt.json"))
nombres = json.load(open("/content/muestras.json"))
cruda = list(np.load("/content/salida_cuantizada.npy", allow_pickle=True))
if len(cruda) == 1 and len(nombres) > 1:
    cruda = list(cruda[0])


def detecciones(img):
    """Un array por clase. Puede venir (N, 5) o (5, max_det) con relleno: leer
    una como la otra no da error, da cero detecciones. Coordenadas en yxyx."""
    out = []
    for ci in range(len(img)):
        arr = np.asarray(img[ci], dtype=np.float32)
        if arr.ndim != 2 or arr.size == 0:
            continue
        if arr.shape[1] != 5 and arr.shape[0] == 5:
            arr = arr.T
        for y1, x1, y2, x2, sc in arr[:, :5]:
            if sc >= CONF:
                out.append((CLASES[ci], float(sc), (x1, y1, x2, y2)))
    return out


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


esperadas, encontradas, total_q = Counter(), Counter(), 0
for i, nombre in enumerate(nombres):
    dets = detecciones(cruda[i])
    total_q += len(dets)
    usadas = set()
    for esp in referencia.get(nombre, []):
        if esp["confianza"] < CONF:
            continue
        esperadas[esp["nombre"]] += 1
        candidatas = [(iou(bb, esp["bbox"]), j) for j, (c, _, bb) in enumerate(dets)
                      if c == esp["nombre"] and j not in usadas]
        if candidatas and max(candidatas)[0] >= IOU_MIN:
            usadas.add(max(candidatas)[1])
            encontradas[esp["nombre"]] += 1

print(f"{'clase':15}{'del .pt':>8}{'conserva':>10}{'recall':>8}")
for c in CLASES:
    if esperadas[c]:
        print(f"{c:15}{esperadas[c]:>8}{encontradas[c]:>10}{encontradas[c] / esperadas[c]:>8.1%}")
total_ref, coinc = sum(esperadas.values()), sum(encontradas.values())
recall = coinc / total_ref if total_ref else 0
precision = coinc / total_q if total_q else 0
print(f"\nRecall {recall:.1%}   Precision {precision:.1%}")
print("OK: la cuantizacion conservo el modelo." if recall >= 0.90 else
      "ATENCION: se perdio algo, probar con cuidado." if recall >= 0.75 else
      "MAL: demasiada perdida. Avisar antes de usarlo en la pista.")

## Paso 10 — Guardar y bajar

Todo queda también en Drive (`jonica26/`). Pasarle estos archivos a quien
actualice el repo, o ponerlos así:

| Archivo | Va en el repo |
|---|---|
| `yolov8n_pare_semaforo_muneco.hef` | `model/` |
| `yolov8n_pare_semaforo_muneco.pt` | `model/` |
| `labels.txt` | `model/` (reemplaza al actual) |
| `muestras.zip` + `referencia_pt.json` | `docs/muestras_muneco/` |

Y en `source/config.py`:

```python
RUTA_HEF = RAIZ / "model" / "yolov8n_pare_semaforo_muneco.hef"
ETIQUETAS = [CLASE_PARE, CLASE_VERDE, CLASE_MUNECO]
```

En la Raspberry, primero que cargue y después que detecte igual que acá:

```bash
hailortcli parse-hef model/yolov8n_pare_semaforo_muneco.hef
python3 tools/verificar_hef.py --imagenes docs/muestras_muneco/muestras --referencia docs/muestras_muneco/referencia_pt.json
```

In [ ]:
exec(open("/content/ajustes.py").read())
import shutil
from pathlib import Path

from google.colab import files

montar_drive()
Path("/content/labels.txt").write_text("".join(c + "\n" for c in CLASES))
!cd /content && rm -f muestras.zip && zip -qr muestras.zip muestras/

for archivo in (HEF, PT, "/content/labels.txt", "/content/referencia_pt.json", "/content/muestras.zip"):
    shutil.copy(archivo, DRIVE)
    files.download(archivo)
print("Copiado en", DRIVE)